In [1]:
import pandas as pd
import requests
# import time

import aiohttp
import asyncio
from io import StringIO
from timeit import default_timer
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
anno = 2002

periods = [

    (f"{anno}-01-01", f"{anno}-03-31"),
    (f"{anno}-04-01", f"{anno}-06-30"),
    (f"{anno}-07-01", f"{anno}-09-30"),
    (f"{anno}-10-01", f"{anno}-12-31"),

    # ("2026-01-01", "2026-03-10")

]

In [68]:
currency = "GBP"

def download_fx(start, end):
    url = f"https://data-api.ecb.europa.eu/service/data/EXR/D.{currency}.EUR.SP00.A"
    params = {"startPeriod": start, "endPeriod": end, "format": "csvdata"}
    r = requests.get(url, params=params)
    df = pd.read_csv(StringIO(r.text))
    return df

results = []

# ThreadPoolExecutor per scaricare tutti i blocchi in parallelo
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(download_fx, start, end) for start, end in periods]
    for f in as_completed(futures):
        results.append(f.result())

# concatena tutti i dataframe
df = pd.concat(results).sort_values("TIME_PERIOD").reset_index(drop=True)

# print(df.head())
print("Totale righe:", len(df))

Totale righe: 261


In [69]:
df = df[['KEY', 'FREQ', 'CURRENCY', 'CURRENCY_DENOM', 'EXR_TYPE', 'EXR_SUFFIX',
       'TIME_PERIOD', 'OBS_VALUE', 'OBS_STATUS', 'OBS_CONF', 'OBS_PRE_BREAK',
       'OBS_COM', 'TIME_FORMAT', 'BREAKS', 'COLLECTION', 'COMPILING_ORG',
       'DISS_ORG', 'DOM_SER_IDS', 'PUBL_ECB', 'PUBL_MU', 'PUBL_PUBLIC',
       'UNIT_INDEX_BASE', 'COMPILATION', 'COVERAGE', 'DECIMALS', 'NAT_TITLE',
       'SOURCE_AGENCY', 'SOURCE_PUB', 'TITLE', 'TITLE_COMPL', 'UNIT',
       'UNIT_MULT']]

In [70]:
df.head()

,KEY,FREQ,CURRENCY,CURRENCY_DENOM,EXR_TYPE,EXR_SUFFIX,TIME_PERIOD,OBS_VALUE,OBS_STATUS,OBS_CONF,...,COMPILATION,COVERAGE,DECIMALS,NAT_TITLE,SOURCE_AGENCY,SOURCE_PUB,TITLE,TITLE_COMPL,UNIT,UNIT_MULT
0,EXR.D.GBP.EUR.SP00.A,D,GBP,EUR,SP00,A,2002-01-01,NaN,H,NaN,...,NaN,NaN,5,NaN,4F0,NaN,Pound sterling/Euro ECB reference exchange rate,"ECB reference exchange rate, Pound sterling/Eu...",GBP,0
1,EXR.D.GBP.EUR.SP00.A,D,GBP,EUR,SP00,A,2002-01-02,0.6262,A,NaN,...,NaN,NaN,5,NaN,4F0,NaN,Pound sterling/Euro ECB reference exchange rate,"ECB reference exchange rate, Pound sterling/Eu...",GBP,0
2,EXR.D.GBP.EUR.SP00.A,D,GBP,EUR,SP00,A,2002-01-03,0.6254,A,NaN,...,NaN,NaN,5,NaN,4F0,NaN,Pound sterling/Euro ECB reference exchange rate,"ECB reference exchange rate, Pound sterling/Eu...",GBP,0
3,EXR.D.GBP.EUR.SP00.A,D,GBP,EUR,SP00,A,2002-01-04,0.6217,A,NaN,...,NaN,NaN,5,NaN,4F0,NaN,Pound sterling/Euro ECB reference exchange rate,"ECB reference exchange rate, Pound sterling/Eu...",GBP,0
4,EXR.D.GBP.EUR.SP00.A,D,GBP,EUR,SP00,A,2002-01-07,0.6194,A,NaN,...,NaN,NaN,5,NaN,4F0,NaN,Pound sterling/Euro ECB reference exchange rate,"ECB reference exchange rate, Pound sterling/Eu...",GBP,0


In [71]:
df.to_csv(f'exchange_rate_GBP_EUR_{anno}.csv')

# Back up